In [1]:
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from torchinfo import summary

### Simple  logit

`nn.Linear(num_features=N, out_features=1)`

$$ z= \mathbf{w}^\top \mathbf{x} + b \quad \text{(logit)} $$
$$ \mathbf{x} \in \mathbb{R}^N, \quad z \in \mathbb{R}^1 $$

`self.sigmoid = nn.Sigmoid()`

$$ \hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}} \in [0,1] $$

`out = self.linear(features)`

$$ \mathcal{M}(\mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b) $$


In [2]:
class Model(nn.Module):         # Inherits nn.Module

    def __init__(self, num_features):

        super().__init__()       # Call parent constructor

        self.linear = nn.Linear(num_features, 1) # Linear layer: num_features → 1 logit
        self.sigmoid = nn.Sigmoid()              # Probability [0,1]

    def forward(self, features):
        
        out = self.linear(features)      # x → w*x + b (logit)
        out = self.sigmoid(out)          # logit → probability

        return out
        

In [3]:
## Create fake data (batch of 10 samples, 5 features each)
features = torch.rand(10, 5) # row=10 (observation), col = 5 (features)

##create model
model = Model(features.shape[1])

##call forward pass

model(features)   ## model.forward(features) << also can do this but dont

tensor([[0.3682],
        [0.3016],
        [0.3071],
        [0.3049],
        [0.3465],
        [0.2885],
        [0.2535],
        [0.3576],
        [0.3609],
        [0.3680]], grad_fn=<SigmoidBackward0>)

In [4]:
model.linear.weight, model.linear.bias

(Parameter containing:
 tensor([[-0.1326, -0.1849, -0.3581, -0.1721, -0.4103]], requires_grad=True),
 Parameter containing:
 tensor([-0.0492], requires_grad=True))

In [5]:
summary(model)

Layer (type:depth-idx)                   Param #
Model                                    --
├─Linear: 1-1                            6
├─Sigmoid: 1-2                           --
Total params: 6
Trainable params: 6
Non-trainable params: 0

In [6]:
summary(model, input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

### Simple Neural Netwrok with hidden network

In [7]:
class Model2(nn.Module):         # Inherits nn.Module

    def __init__(self, num_features):

        super().__init__()       # Call parent constructor

        self.linear1 = nn.Linear(num_features, 3) # Linear layer: num_features → 3 node
        self.relu = nn.ReLU()                     # Relu activation in hidden layer
        self.linear2 = nn.Linear(3, 1)            # Linear layer: num_features → 1 logit
        self.sigmoid = nn.Sigmoid()               # Probability [0,1]

    def forward(self, features):
        
        out = self.linear1(features)     
        out = self.relu(out)
        out = self.linear2(out) 
        out = self.sigmoid(out)         
        
        return out
        

In [8]:
## Create fake data (batch of 10 samples, 5 features each)
features = torch.rand(10, 5) # row=10 (observation), col = 5 (features)

##create model
model2 = Model2(features.shape[1])

##call forward pass

model2(features) 

tensor([[0.6074],
        [0.5975],
        [0.6028],
        [0.5971],
        [0.6258],
        [0.6110],
        [0.6065],
        [0.6309],
        [0.6279],
        [0.6059]], grad_fn=<SigmoidBackward0>)

In [9]:
model2.linear1.weight, model2.linear2.weight, model2.linear1.bias, model2.linear2.bias

(Parameter containing:
 tensor([[ 0.3749, -0.2175,  0.0641, -0.2100, -0.1822],
         [-0.3339, -0.1470,  0.2638, -0.2086,  0.1717],
         [-0.3784,  0.3124,  0.0710,  0.1907,  0.4066]], requires_grad=True),
 Parameter containing:
 tensor([[-0.4221, -0.3364,  0.1870]], requires_grad=True),
 Parameter containing:
 tensor([ 0.0506,  0.2234, -0.1906], requires_grad=True),
 Parameter containing:
 tensor([0.4833], requires_grad=True))

In [10]:
summary(model2, input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model2                                   [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

### Use of Sequential container

In [11]:
class Model3(nn.Module):              # Inherits nn.Module

    def __init__(self, num_features):

        super().__init__()            # Call parent constructor

        self.network = nn.Sequential(
            nn.Linear(num_features, 3),    # Linear layer: num_features → 3 node
            nn.ReLU(),                     # Relu activation in hidden layer
            nn.Linear(3, 1),               # Linear layer: num_features → 1 logit
            nn.Sigmoid()                   # Probability [0,1]
        )
        
    def forward(self, features):
        
        out = self.network(features)              
        
        return out
        

In [12]:
## Create fake data (batch of 10 samples, 5 features each)
features = torch.rand(10, 5) # row=10 (observation), col = 5 (features)

##create model
model3 = Model3(features.shape[1])

##call forward pass

model2(features) 

tensor([[0.6307],
        [0.6071],
        [0.6017],
        [0.5984],
        [0.6183],
        [0.5798],
        [0.6100],
        [0.6173],
        [0.5875],
        [0.6060]], grad_fn=<SigmoidBackward0>)

In [13]:
summary(model3, input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model3                                   [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

---

In [14]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

In [15]:
### binary clasification of breast cancer
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [16]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)


X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [54]:
### NN in pytorch

class SimpleNN(nn.Module):

    def __init__(self, num_features): 

        super().__init__()

        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)

        return out
        

In [62]:
learning_rate = 0.1
epochs = 5000
loss_function = nn.BCELoss()

In [65]:
# create model
model = SimpleNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

X_train_tensor = X_train_tensor.float()   # Convert Double → Float
y_train_tensor = y_train_tensor.float()   # Same for targets

for epoch in tqdm(range(epochs)):
    
    # forward pass
    y_pred = model(X_train_tensor)
    
    # loss calculate
    loss = loss_function(y_pred.squeeze(), y_train_tensor)

    # make zero gradients as it gets accumulated
    optimizer.zero_grad()
    
    # backward pass
    loss.backward()

    # update weights and biases
    optimizer.step()

    if epoch % 500 == 0 or epoch == epochs-1:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


 47%|████████████████████████▋                            | 2334/5000 [00:00<00:00, 11873.57it/s]

Epoch 0, Loss: 0.6177
Epoch 500, Loss: 0.0655
Epoch 1000, Loss: 0.0582
Epoch 1500, Loss: 0.0548
Epoch 2000, Loss: 0.0528


100%|█████████████████████████████████████████████████████| 5000/5000 [00:00<00:00, 12355.99it/s]

Epoch 2500, Loss: 0.0513
Epoch 3000, Loss: 0.0502
Epoch 3500, Loss: 0.0493
Epoch 4000, Loss: 0.0485
Epoch 4500, Loss: 0.0478
Epoch 4999, Loss: 0.0472


In [66]:
X_test_tensor = X_test_tensor.float()
y_test_tensor = y_train_tensor.float()

# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5332562327384949


In [58]:
# y_pred:         [455, 1]  ← model.linear(..., 1) + sigmoid
# y_pred.squeeze(): [455]   ← removes dim=1 (size 1)
# y_train_tensor: [455]     ← perfect match!

# BCELoss >> [455] vs [455]

y_pred.shape, y_pred.squeeze().shape

(torch.Size([114, 1]), torch.Size([114]))

In [49]:
model.parameters??

Signature: model.parameters(recurse: bool = True) -> Iterator[torch.nn.parameter.Parameter]
Source:   
    def parameters(self, recurse: bool = True) -> Iterator[Parameter]:
        r"""Return an iterator over module parameters.

        This is typically passed to an optimizer.

        Args:
            recurse (bool): if True, then yields parameters of this module
                and all submodules. Otherwise, yields only parameters that
                are direct members of this module.

        Yields:
            Parameter: module parameter

        Example::

            >>> # xdoctest: +SKIP("undefined vars")
            >>> for param in model.parameters():
            >>>     print(type(param), param.size())
            <class 'torch.Tensor'> (20L,)
            <class 'torch.Tensor'> (20L, 1L, 5L, 5L)

        """
        for name, param in self.named_parameters(recurse=recurse):
            yield param
File:      ~/miniconda3/envs/pytorch_env/lib/python3.10/site-packages/torc